In [ ]:
import os, glob, subprocess

# Discover the SQLite DB path at runtime — handles any Kaggle mount variation
# Dataset attached as: utkarshpatelthefirst/master-data-1min-db
print("=== Kaggle Input Directory Contents ===")
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        fpath = os.path.join(root, f)
        fsize = os.path.getsize(fpath) / (1024**3)
        print(f"  {fpath}  ({fsize:.2f} GB)")

# Find the SQLite file
hits = glob.glob('/kaggle/input/**/*.sqlite', recursive=True)
if not hits:
    raise FileNotFoundError("No .sqlite file found under /kaggle/input — check dataset attachment!")

DB_PATH = hits[0]
print(f"\n✅ Using DB: {DB_PATH}")

# Stage 1 — NSE 500 Pairs Trading: Pearson Correlation Screening
**Methodology:** Compute pairwise Pearson correlation of session-continuous log-returns for all NSE 500 equities using 1-min OHLCV data.
**Key design decisions:**
- Log-returns only (`ln(close_t / close_{t-1})`), never raw prices
- Market-hours filter: 09:15–15:29 IST only (no overnight/weekend gaps)
- Session-open bars (09:15) are nulled: overnight return is NOT an intraday return
- Inner-join timestamp alignment across all symbols
- t-stat + p-value computed per pair; filter p < 0.05
- Outputs: `pairs_top500.csv` and `pairs_all.csv`

In [ ]:
# Install / verify dependencies (scipy needed for t-distribution p-values)
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scipy"], check=True)
print("Dependencies ready")

## Stage 1 — Load Raw Data from SQLite
**Input:** `/kaggle/input/master-data-1min-db/Master-Data-1min.sqlite` (ohlcv_1min table)
**Output:** `df` — raw DataFrame with columns [symbol, timestamp, close]
**Core Logic:** Single SQL query selecting only the columns we need; ORDER BY timestamp for correctness.
**Formula:** No formula — data extraction step.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import datetime, warnings
warnings.filterwarnings('ignore')

# DB_PATH set by the path-discovery cell above
con = sqlite3.connect(DB_PATH)
df = pd.read_sql(
    "SELECT symbol, timestamp, close FROM ohlcv_1min ORDER BY timestamp",
    con
)
con.close()

print(f"Raw rows loaded  : {len(df):,}")
print(f"Unique symbols   : {df['symbol'].nunique()}")
print(f"Timestamp range  : {df['timestamp'].min()} → {df['timestamp'].max()}")
print(f"Close dtype      : {df['close'].dtype}")
print(f"Sample:\n{df.head(3)}")

## Stage 2 — Filter to NSE Market Hours (Session-Continuous Series)
**Input:** `df` — raw OHLCV rows
**Output:** `df_trading` — rows restricted to 09:15–15:29 IST only
**Core Logic:**
1. Convert UNIX epoch timestamp → pandas Timestamp in IST (Asia/Kolkata, UTC+5:30)
2. Extract time-of-day component
3. Keep only rows where 09:15 ≤ time ≤ 15:29 — discards overnight, weekends, holidays
4. Result: all rows represent genuine intraday 1-minute bars
**Formula:** No formula — temporal filter step.

In [ ]:
# Convert UNIX epoch (seconds) → IST datetime
# Fyers stores timestamps as UNIX epoch integers in UTC
df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert('Asia/Kolkata')
df['time_only'] = df['dt'].dt.time
df['date']      = df['dt'].dt.date

MARKET_OPEN  = datetime.time(9, 15)
MARKET_CLOSE = datetime.time(15, 29)

# Keep only intraday bars (strict market hours filter)
df_trading = df[
    (df['time_only'] >= MARKET_OPEN) &
    (df['time_only'] <= MARKET_CLOSE)
].copy()

bars_per_sym = len(df_trading) // df_trading['symbol'].nunique()
print(f"After market-hours filter : {len(df_trading):,} rows")
print(f"Approx bars per symbol    : {bars_per_sym:,}")
print(f"Approx trading days       : {bars_per_sym // 375}")
print(f"Expected bars/day         : 375  (09:15 through 15:29 = 375 bars)")

## Stage 3 — Pivot to Price Matrix & Align Timestamps (Inner Join)
**Input:** `df_trading`
**Output:** `price_matrix` — shape (n_common_timestamps × n_symbols), zero NaN
**Core Logic:**
1. Pivot DataFrame: each row = one timestamp, each column = one symbol's close price
2. Inner join: drop any timestamp not shared by ALL symbols (ensures perfect alignment)
3. Verify no NaN or non-positive prices remain
**Formula:** No formula — pivot + inner-join alignment.

In [ ]:
# Pivot: index = IST datetime, columns = symbol
price_matrix = df_trading.pivot(index='dt', columns='symbol', values='close')
n_total_bars = len(price_matrix)
print(f"Pivot shape (all timestamps) : {price_matrix.shape}")
print(f"NaN count before alignment   : {price_matrix.isnull().sum().sum():,}")

# ── Smart two-pass alignment ─────────────────────────────────────────────────
# Pass 1: Drop symbols with < 80% coverage (sparse listings / recent IPOs)
#         These would collapse the inner-join to a tiny window
coverage = price_matrix.notna().sum() / n_total_bars
sparse_symbols = coverage[coverage < 0.80].index.tolist()
if sparse_symbols:
    print(f"Dropping {len(sparse_symbols)} sparse symbols (<80% coverage): {sparse_symbols[:10]}...")
    price_matrix = price_matrix.drop(columns=sparse_symbols)
print(f"After sparse-symbol drop     : {price_matrix.shape}")

# Pass 2: Inner join on remaining symbols — drop timestamps missing ANY survivor
price_matrix = price_matrix.dropna(how='any', axis=0)
print(f"After inner-join alignment   : {price_matrix.shape}")
print(f"  → {price_matrix.shape[0]:,} common bars × {price_matrix.shape[1]} symbols")
print(f"  → {price_matrix.shape[0] / 374:.1f} effective trading days retained")

# Data quality assertions
assert price_matrix.isnull().sum().sum() == 0, "NaN still present after alignment"
assert (price_matrix > 0).all().all(), "Non-positive prices found — data quality issue"
assert price_matrix.shape[0] >= 5000, f"Too few bars after alignment ({price_matrix.shape[0]}) — check data"

print("\nPrice matrix sanity checks: PASSED ✅")
print(f"Sample (first 3 symbols, first 3 bars):")
print(price_matrix.iloc[:3, :3])

## Stage 4 — Compute Session-Continuous Log-Returns
**Input:** `price_matrix`
**Output:** `log_returns` — same shape minus session-open bars and shift row
**Core Logic:**
1. Compute `ln(close_t / close_{t-1})` across all symbols simultaneously (vectorised)
2. **Null every session-open bar (09:15)**: its return spans overnight → NOT an intraday return
3. Drop all rows containing NaN (the shift-NaN first row + all nulled 09:15 rows)
4. Result: a clean, gap-free return series containing only genuine intraday 1-min returns

**Formula:**
$$r_t = \ln\left(\frac{P_t}{P_{t-1}}\right) \quad \text{for } t \text{ within the same session only}$$

NSE: 375 bars/day → 374 intraday returns/day (09:15 bar dropped). Over 120 trading days → ~44,880 clean returns per symbol.

In [ ]:
# Step 1: Raw log-returns (includes overnight returns at session boundaries)
log_returns_raw = np.log(price_matrix / price_matrix.shift(1))
print(f"Raw log-returns shape        : {log_returns_raw.shape}")
print(f"NaN count (expected = n_syms): {log_returns_raw.isnull().sum().sum()}")

# Step 2: Null session-open bars (09:15 returns are overnight returns — discard)
session_open_mask = (price_matrix.index.time == MARKET_OPEN)
n_session_opens = session_open_mask.sum()
log_returns_raw[session_open_mask] = np.nan
print(f"Session-open bars nulled     : {n_session_opens:,} rows × {price_matrix.shape[1]} symbols")

# Step 3: Drop all NaN rows (shift row + all session-open rows)
log_returns = log_returns_raw.dropna(how='any')

# Step 4: Sanity checks
assert log_returns.isnull().sum().sum() == 0,      "❌ NaN remains in log_returns"
assert not np.isinf(log_returns.values).any(),     "❌ Inf in log_returns — zero price detected"
assert (log_returns.abs() < 1.0).all().all(),      "❌ |return| >= 100% — likely a data error"

n_bars    = len(log_returns)
n_symbols = log_returns.shape[1]
print(f"\nClean log-return matrix      : {n_bars:,} bars × {n_symbols} symbols")
print(f"Expected approx              : ~{120 * 374:,} bars (120 days × 374 returns/day)")
print(f"Return range: [{log_returns.values.min():.6f}, {log_returns.values.max():.6f}]")
print("Log-return sanity checks: PASSED ✅")

## Stage 5 — Pearson Correlation Matrix
**Input:** `log_returns` — clean intraday return matrix
**Output:** `corr_df` — (n_symbols × n_symbols) Pearson correlation matrix
**Core Logic:**
1. Attempt GPU-accelerated correlation via cuDF (Kaggle T4 has ~15GB GPU RAM)
2. Fall back to CPU NumPy (BLAS-optimised LAPACK, perfectly adequate for 500×45k matrix)
3. Scale-invariance: multiplying returns by any constant leaves ρ unchanged — no pre-scaling needed

**Formula:**
$$\rho_{A,B} = \frac{\sum_t (r_t^A - \bar{r}^A)(r_t^B - \bar{r}^B)}{\sqrt{\sum_t (r_t^A - \bar{r}^A)^2 \cdot \sum_t (r_t^B - \bar{r}^B)^2}}$$

In [ ]:
print("Computing Pearson correlation matrix...")
import time
t0 = time.time()

try:
    import cudf
    # Use cudf.DataFrame() constructor directly — from_pandas() removed in newer cuDF
    lr_gpu  = cudf.DataFrame(log_returns)
    corr_df = lr_gpu.corr().to_pandas()
    print(f"✅ GPU correlation complete in {time.time()-t0:.1f}s")
    backend = "cuDF (GPU)"
except Exception as gpu_err:
    # Broad catch: handles ImportError, AttributeError, MemoryError, etc.
    print(f"GPU path failed ({type(gpu_err).__name__}: {gpu_err}) — using CPU")
    corr_df = log_returns.corr(method='pearson')
    print(f"✅ CPU correlation complete in {time.time()-t0:.1f}s")
    backend = "pandas (CPU)"

print(f"Correlation matrix shape : {corr_df.shape}")
print(f"Backend used             : {backend}")
print(f"Diagonal check (should all be 1.0): min={np.diag(corr_df.values).min():.6f}, max={np.diag(corr_df.values).max():.6f}")
print(f"Off-diagonal range       : [{np.nanmin(corr_df.values[corr_df.values < 0.9999]):.4f}, {np.nanmax(corr_df.values[corr_df.values < 0.9999]):.4f}]")

## Stage 6 — Extract Pairs, Compute t-statistic & p-value, Rank
**Input:** `corr_df`, `n_bars` (number of aligned observations)
**Output:** `pairs_df` — ranked DataFrame of all statistically significant pairs
**Core Logic:**
1. Iterate upper triangle of correlation matrix (avoids duplicate A-B / B-A and self-pairs A-A)
2. For each pair compute t-statistic and two-tailed p-value
3. Filter: p < 0.05 AND n_obs >= 5,000
4. Sort descending by pearson_rho; assign rank

**Formula:**
$$t = \rho \cdot \sqrt{\frac{n-2}{1-\rho^2}} \sim t_{n-2} \quad \text{under } H_0: \rho = 0$$

Total pairs for N=500: $\binom{500}{2} = 124{,}750$

In [ ]:
from scipy.stats import t as t_dist

symbols   = corr_df.columns.tolist()
n_sym     = len(symbols)
n_obs     = n_bars   # all pairs share the same aligned observation count
MIN_OBS   = 5_000
ALPHA     = 0.05

corr_vals = corr_df.values  # numpy array for speed
rows      = []

print(f"Extracting pairs from {n_sym}×{n_sym} matrix...")
print(f"Total upper-triangle pairs : {n_sym*(n_sym-1)//2:,}")
t0 = time.time()

for i in range(n_sym):
    for j in range(i + 1, n_sym):
        rho = corr_vals[i, j]
        if np.isnan(rho):
            continue
        # Clamp to [-1,1] to guard against floating-point drift at ±1
        rho = float(np.clip(rho, -0.999999, 0.999999))
        # t-statistic under H0: rho=0
        t_stat = rho * np.sqrt((n_obs - 2) / (1.0 - rho**2))
        # Two-tailed p-value
        p_val  = 2.0 * t_dist.sf(abs(t_stat), df=n_obs - 2)
        rows.append((symbols[i], symbols[j], rho, t_stat, p_val, n_obs))

print(f"Pair loop done in {time.time()-t0:.1f}s")

pairs_df = pd.DataFrame(rows, columns=[
    'symbol_a', 'symbol_b', 'pearson_rho', 't_stat', 'p_value', 'n_obs'
])
print(f"Total pairs before filter : {len(pairs_df):,}")

# Filter: statistical significance + minimum observations
pairs_df = pairs_df[
    (pairs_df['p_value'] < ALPHA) &
    (pairs_df['n_obs'] >= MIN_OBS)
].copy()
print(f"After p<0.05 & n≥5000    : {len(pairs_df):,}")

# Sort descending by pearson_rho (highest correlation first)
pairs_df = pairs_df.sort_values('pearson_rho', ascending=False).reset_index(drop=True)
pairs_df['rank'] = pairs_df.index + 1

# Round for clean CSV output
pairs_df['pearson_rho'] = pairs_df['pearson_rho'].round(6)
pairs_df['t_stat']      = pairs_df['t_stat'].round(4)
pairs_df['p_value']     = pairs_df['p_value'].round(8)

print(f"\nTop 5 pairs:")
print(pairs_df.head(5).to_string(index=False))
print(f"\nBottom 5 pairs (by ρ):")
print(pairs_df.tail(5).to_string(index=False))

## Stage 7 — Validation Checklist
**Input:** `pairs_df`, `log_returns`, `price_matrix`
**Output:** Pass/Fail assertions — notebook aborts on any failure
**Core Logic:** Run all pre-acceptance checks before exporting CSVs.
**Formula:** No formula — validation step.

In [ ]:
print("Running validation checklist...\n")

checks = []

# 1. No NaN in log_returns
v = log_returns.isnull().sum().sum() == 0
checks.append(("No NaN in log_returns", v))

# 2. No Inf in log_returns
v = not np.isinf(log_returns.values).any()
checks.append(("No Inf in log_returns", v))

# 3. pearson_rho strictly in (-1, 1)
v = pairs_df['pearson_rho'].between(-1, 1, inclusive='neither').all()
checks.append(("All pearson_rho in (-1, 1)", v))

# 4. All p_values < 0.05
v = (pairs_df['p_value'] < 0.05).all()
checks.append(("All p_value < 0.05", v))

# 5. All n_obs >= 5000
v = (pairs_df['n_obs'] >= 5000).all()
checks.append(("All n_obs >= 5,000", v))

# 6. rank is 1-indexed contiguous
v = pairs_df['rank'].tolist() == list(range(1, len(pairs_df)+1))
checks.append(("rank is 1-indexed contiguous", v))

# 7. No duplicate pairs (symbol_a < symbol_b always)
v = (pairs_df['symbol_a'] < pairs_df['symbol_b']).all()
checks.append(("symbol_a always < symbol_b (no dupe pairs)", v))

# 8. At least 500 pairs for top-500 export
v = len(pairs_df) >= 500
checks.append(("At least 500 valid pairs exist", v))

# 9. All n_obs equal (strict alignment used)
v = pairs_df['n_obs'].nunique() == 1
checks.append(("All pairs share same n_obs (strict alignment)", v))

# 10. Price matrix had zero NaN after alignment
v = price_matrix.isnull().sum().sum() == 0
checks.append(("Price matrix zero NaN after alignment", v))

print(f"{'Check':<45} {'Result'}")
print("-" * 55)
all_passed = True
for name, passed in checks:
    icon = "✅" if passed else "❌"
    print(f"{name:<45} {icon}")
    if not passed:
        all_passed = False

print()
assert all_passed, "❌ Validation failed — see above. Do NOT export CSVs."
print("🎉 All validation checks PASSED — safe to export.")

## Stage 8 — Export CSVs
**Input:** `pairs_df`
**Output:** `pairs_all.csv`, `pairs_top500.csv` saved to `/kaggle/working/`
**Formula:** No formula — I/O step.

In [ ]:
import os

out_dir = '/kaggle/working'

# All pairs
all_path  = os.path.join(out_dir, 'pairs_all.csv')
top_path  = os.path.join(out_dir, 'pairs_top500.csv')

pairs_df.to_csv(all_path, index=False)
pairs_df.head(500).to_csv(top_path, index=False)

all_size  = os.path.getsize(all_path)  / 1024
top_size  = os.path.getsize(top_path)  / 1024

print(f"pairs_all.csv    : {len(pairs_df):>8,} rows  |  {all_size:>8.1f} KB")
print(f"pairs_top500.csv :        500 rows  |  {top_size:>8.1f} KB")
print(f"\nTop 500 ρ range  : {pairs_df.iloc[0]['pearson_rho']:.4f} → {pairs_df.iloc[499]['pearson_rho']:.4f}")
print(f"Overall ρ range  : {pairs_df.iloc[0]['pearson_rho']:.4f} → {pairs_df.iloc[-1]['pearson_rho']:.4f}")

## Stage 9 — Publish as Kaggle Dataset
**Input:** Both CSV files in `/kaggle/working/`
**Output:** Published dataset `utkarshpatelthefirst/pairs-stage1-pearson` on Kaggle
**Formula:** No formula — dataset publishing step.

In [ ]:
import json, shutil
from kaggle.api.kaggle_api_extended import KaggleApi

# Hardcoded credentials (correct for Kaggle environment — no ~/.quant_env available)
os.environ['KAGGLE_USERNAME'] = 'utkarshpatelthefirst'
os.environ['KAGGLE_KEY']      = 'fbef16329099428205f671dd5de8337b'

api = KaggleApi()
api.authenticate()

export_dir = '/kaggle/working/dataset_export'
os.makedirs(export_dir, exist_ok=True)

# Copy both CSVs
shutil.copy(all_path,  os.path.join(export_dir, 'pairs_all.csv'))
shutil.copy(top_path,  os.path.join(export_dir, 'pairs_top500.csv'))

# Write dataset metadata
meta = {
    "title"    : "Pairs Stage1 Pearson",
    "id"       : "utkarshpatelthefirst/pairs-stage1-pearson",
    "licenses" : [{"name": "CC0-1.0"}]
}
with open(os.path.join(export_dir, 'dataset-metadata.json'), 'w') as f:
    json.dump(meta, f, indent=2)

print("Publishing dataset...")
api.dataset_create_new(export_dir, dir_mode='zip', quiet=False)
print("\n✅ Dataset published!")
print("   URL: https://www.kaggle.com/datasets/utkarshpatelthefirst/pairs-stage1-pearson")